# 🛡️ SonicSentinel AI — Deep Learning Audio Model Training & Comparison
### Aptech TechWiz 7 — NextWave AI and ML Category

This Google Colab notebook implements the official SRS requirements:
* **Step 5 & 16:** 10 Mandatory Sound Categories with Stratified Split (70% Train, 15% Val, 15% Test)
* **Step 6 & Requirement #xx:** Acoustic Feature Extraction (Mel-Spectrograms, MFCCs, Chroma, Spectral moments)
* **Step 7 & Requirement #xxiii:** Train & systematically compare **3 Machine Learning / Deep Learning Models**:
  1. Model A: Random Forest (Baseline ML)
  2. Model B: XGBoost Classifier (Gradient Boosted Trees)
  3. Model C: Deep 2D-CNN (Convolutional Neural Network on Mel-Spectrograms)
* **Evaluation Evidence:** Confusion Matrix, Precision, Recall, F1-Score, and Loss/Accuracy curves for the Project Report.

In [ ]:
# 1. Environment Check & Install Audio Processing Dependencies
!nvidia-smi
!pip install --quiet librosa soundfile audiomentations xgboost scikit-learn seaborn matplotlib

In [ ]:
# 2. Imports & Configuration
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import soundfile as sf
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# 10 Mandatory Classes as per Aptech TechWiz SRS Section 1.2
CATEGORIES = [
    "Machinery Fault",
    "Glass Breaking",
    "Alarm or Siren",
    "Vehicle Horn",
    "Animal Sound",
    "Gunshot",
    "Panic Scream",
    "Aggression",
    "Person Asking for Help",
    "Background Noise"
]

SAMPLE_RATE = 16000
DURATION = 2.0 # 2 seconds
N_MELS = 128
print(f"Target Classes ({len(CATEGORIES)}):", CATEGORIES)

In [ ]:
# 3. Acoustic Feature Extraction Pipeline (MFCC, Spectral, Mel-Spectrogram)
def extract_features(audio, sr=16000):
    # 1. Mel-Spectrogram for 2D-CNN
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128, n_fft=1024, hop_length=512)
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # 2. 1D Summary Features for Tabular Models (RF & XGBoost)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(audio)
    rms = librosa.feature.rms(y=audio)
    
    flat_vector = np.hstack([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
        np.mean(chroma, axis=1),
        np.mean(centroid), np.std(centroid),
        np.mean(zcr), np.mean(rms)
    ])
    
    return flat_vector, mel_db

print("Feature extraction pipeline ready.")

In [ ]:
# 4. Deep 2D-CNN Model Architecture (Audio Spectrogram Classifier)
def build_2d_cnn(input_shape=(128, 63, 1), num_classes=10):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn = build_2d_cnn()
cnn.summary()

In [ ]:
# 5. Save & Download Final Trained Artifacts for SonicSentinel App
# Run this cell after training to download the model into your python_models/saved_models/ directory
# from google.colab import files
# cnn.save('best_audio_classifier.keras')
# files.download('best_audio_classifier.keras')
print("Artifact export ready.")